# 第三章：分配角色（角色提示）

- [课程](#lesson)
- [练习](#exercises)
- [示例实验区](#example-playground)

## 设置

运行以下设置单元格以加载您的 API 密钥并建立 `get_completion` 辅助函数。

In [ ]:
# Import python's built-in regular expression library
import re
import boto3
import json

# Import the hints module from the utils package
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Retrieve the MODEL_NAME variable from the IPython store
%store -r MODEL_NAME
%store -r AWS_REGION

client = boto3.client('bedrock-runtime',region_name=AWS_REGION)

def get_completion(prompt,system=''):
    body = json.dumps(
        {
            "anthropic_version": '',
            "max_tokens": 2000,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.0,
            "top_p": 1,
            "system": system
        }
    )
    response = client.invoke_model(body=body, modelId=MODEL_NAME)
    response_body = json.loads(response.get('body').read())

    return response_body.get('content')[0].get('text')

---

## 课程

继续 Claude 除了您所说的内容之外没有上下文这一主题，有时**提示 Claude 扮演特定角色（包括所有必要的上下文）**很重要。这也称为角色提示。角色上下文的详细程度越高，效果越好。

**为 Claude 设定角色可以提高 Claude 在各个领域的性能**，从写作到编码再到摘要。这就像人类有时在被告知"像 ______ 一样思考"时会有所帮助一样。角色提示还可以改变 Claude 响应的风格、语气和方式。

**注意：** 角色提示可以在系统提示中进行，也可以作为 User 消息轮次的一部分。

### 示例

在下面的例子中，我们看到在没有角色提示的情况下，当被问及对滑板的单句看法时，Claude 提供了**直接且无风格化的答案**。

然而，当我们让 Claude 扮演猫的角色时，Claude 的观点发生了变化，因此 **Claude 的响应语气、风格、内容都适应了新角色**。

**注意：** 您可以使用的一个额外技巧是**为 Claude 提供关于其目标受众的上下文**。下面，"你是一只猫"产生的响应与"你是一只对一群滑板手讲话的猫"会产生完全不同的响应。

这是没有在系统提示中进行角色提示的提示：

In [ ]:
# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Print Claude's response
print(get_completion(PROMPT))

这是相同的用户问题，但使用了角色提示。

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a cat."

# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

您可以使用角色提示作为一种方式来让 Claude 模仿某些写作风格，以某种声音说话，或者引导其答案的复杂性。**角色提示还可以使 Claude 更好地执行数学或逻辑任务。**

例如，在下面的例子中，有一个明确的正确答案，即是。然而，Claude 答错了，认为它缺少信息，但实际上它并没有：

In [ ]:
# Prompt
PROMPT = "Jack is looking at Anne. Anne is looking at George. Jack is married, George is not, and we don’t know if Anne is married. Is a married person looking at an unmarried person?"

# Print Claude's response
print(get_completion(PROMPT))

现在，如果我们**让 Claude 充当逻辑机器人**怎么样？这将如何改变 Claude 的答案？

事实证明，通过这个新的角色分配，Claude 答对了。（虽然值得注意的是，并非所有原因都是正确的）

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a logic bot designed to answer complex logic problems."

# Prompt
PROMPT = "Jack is looking at Anne. Anne is looking at George. Jack is married, George is not, and we don’t know if Anne is married. Is a married person looking at an unmarried person?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

**注意：** 您将在本课程中学到的是，**有许多提示工程技术可用于获得类似的结果**。您使用哪些技术取决于您和您的偏好！我们鼓励您**尝试找到自己的提示工程风格**。

如果您想在不更改上面任何内容的情况下尝试课程提示，请滚动到课程 notebook 的最底部访问[**示例实验区**](#example-playground)。

---

## 练习
- [练习 3.1 - 数学纠正](#exercise-31---math-correction)

### 练习 3.1 - 数学纠正
在某些情况下，**Claude 可能在数学方面遇到困难**，即使是简单的数学。下面，Claude 错误地评估数学问题为正确求解，即使第二步中存在明显的算术错误。请注意，Claude 在逐步检查时实际上发现了错误，但没有得出整体解决方案是错误的结论。

修改 `PROMPT` 和/或 `SYSTEM_PROMPT` 使 Claude 将解决方案评为`不正确`求解，而不是正确求解。


In [ ]:
# System prompt - if you don't want to use a system prompt, you can leave this variable set to an empty string
SYSTEM_PROMPT = ""

# Prompt
PROMPT = """Is this equation solved correctly below?

2x - 3 = 9
2x = 6
x = 3"""

# Get Claude's response
response = get_completion(PROMPT, SYSTEM_PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    if "incorrect" in text or "not correct" in text.lower():
        return True
    else:
        return False

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ 如果您需要提示，请运行下面的单元格！

In [ ]:
print(hints.exercise_3_1_hint)

### 恭喜！

如果您已经解决了到目前为止的所有练习，那么您就可以进入下一章了。祝您提示愉快！

---

## 示例实验区

这是一个让您自由尝试本课程中显示的提示示例并调整提示以查看它如何影响 Claude 响应的区域。

In [ ]:
# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a cat."

# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

In [ ]:
# Prompt
PROMPT = "Jack is looking at Anne. Anne is looking at George. Jack is married, George is not, and we don’t know if Anne is married. Is a married person looking at an unmarried person?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a logic bot designed to answer complex logic problems."

# Prompt
PROMPT = "Jack is looking at Anne. Anne is looking at George. Jack is married, George is not, and we don’t know if Anne is married. Is a married person looking at an unmarried person?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))